# S1 2024 COMPSCI 714 - Tutorial 2: Introduction to Neural Networks

Welcome to Tutorial 2! This tutorial covers basics of neural networks training and evaluation using Keras/Tensorflow and PyTorch. 

*Disclaimer: some parts of the code and text used in this Notebook is directly reused or adapted from Aurélien Géron's notebook: https://github.com/ageron/handson-ml3/blob/main/02_end_to_end_machine_learning_project.ipynb and his book "Hands-on Machine Learning with Scikit-Learn, Keras and Tensorflow, Ed.3", more particulary from Chapter 10.*



**Important note following Tutorial 1**: In Tutorial 1, you used Pandas and Scikit-Learn to load and explore your data as well as to build pre-processing pipelines. You also saw briefly how to import a dataset using TensorFlow and PyTorch.  
When training large models on large datasets using Keras/TensorFlow, you might want to use the dedicated API provided to build complex and efficient data loading and pre-processing pipeline: [`tf.data`](https://www.tensorflow.org/guide/data). Chapter 13 of the *"Hands-on Machine Learning with Scikit-Learn, Keras and Tensorflow, Ed.3"* book is dedicated to this API.  
If you are using PyTorch, note that the [`DataLoader` and `Dataset` classes](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html) are provided to faciliate data loading. As part of the PyTorch project, the Beta library [`torchdata`](https://pytorch.org/data/beta/index.html) is also provided to build flexible and performant pre-processing pipelines. On top of this, you also have access to specific libraries to load and pre-process image ([`torchvision`](https://pytorch.org/vision/stable/index.html)), audio ([`torchaudio`](https://pytorch.org/audio/stable/index.html)), and text ([`torchtext`](https://pytorch.org/text/stable/index.html)) data.  
We will not have time to explore all of these in this course, but feel free to explore the links depending on your interests and needs. 

Let's now get started with Tutorial 2!  
First of all, we need to import some libraries we will use in this tutorial:

In [ ]:
import os
from datetime import datetime
import numpy as np # Scientific computing library
import sklearn # Machine Learning library
import matplotlib.pyplot as plt # Visualisation library

In [ ]:
# Loading and manipulating data
import pandas as pd
from pathlib import Path
import tarfile
import urllib.request

In [ ]:
# TensorFlow
import tensorflow as tf

In [ ]:
# Pytorch
import torch

## I. A Single Artificial Neuron: The Perceptron

### I.1. Introduction to the Perceptron

The Perceptron is one of the earliest model inspired by human biological neurons. It takes several input numbers and output one number. Let's call the output $z$ and the inputs matrix $X$. The output is calculated as:  
$$
z = g(W^TX+b)
\tag{1}
$$
where $W$ is the matrix of weights applied to the inputs, $b$ the bias term, and $step()$ the step activation function applied to the weighted sum of the inputs.  
The matrix of inputs $X$ has one row per instance and one column per predictive attribute.  
The matrix of weights $W$ has one row per input's predictive attribute, and one column in the case of a single neuron Perceptron.  
The $g()$ function is the activation function. This function is used to calculate the output of the neuron. It is sometimes called *transfer function*. In a Perceptron, the $step()$ function is used as activation. Two of the most common step functions are the heaviside function and the sign function, usually centred on 0.  
The bias $b$ is a single value in the case of a single neuron Perceptron. Sometimes, $b$ is integrated in the $W$ matrix as a weight associated to a constant input equal to 1. In that case, the equation above simplifies to:
$$
z = g(W^TX)
\tag{2}
$$
with $X[0] = 1$ and $W[0] = b$.

<figure>
    <center> <img src="img/Perceptron.jpg"  alt='perceptron' width="300"  ><center/>
    <figcaption>A Perceptron with $n$ inputs (+ 1 for the bias term). Notation equivalences: $x_i \equiv X[i]$ and $w_i \equiv W[i]$.</figcaption>
<figure/>

$W$ and $b$ are *trainable* parameters. The parameters are trained by updating their values in function of the difference between the observed output value for an example and the actual output value given the current values of the weights. This is represented in the following equation:
$$
w_{i}^{t+1} = w_{i}^{t} + \mu(y-\hat{y})x_i
\tag{3}
$$
with $w_{i}$ the value of the weight associated to a specific attribute $x_i$, $w_{i}^{t}$ the current value of the weight, $w_{i}^{t+1}$ the updated value of the weight, $y$ the expected output value (i.e., target) and $\hat{y}$ the observed output value.  
$\mu$ is the learning rate. It is an hyperparameter controling the size of the weight update, i.e. the rate of learning. You do not have to worry about it now, we will discuss more about it in Lecture 5. 

An important observation about the single neuron Perceptron is that it can only model linearly seperable data. You can see that by looking at Equation (1). The output is function of a linear combination of the inputs, and the activation function is a step function. Therefore, the decision boundary modelled by a single neuron Perceptron can only be linear. We will discuss how more complex neural networks can model non-linear patterns using non-linear activation functions more in details in Lecture 5. 

### I.2. The Iris dataset

To experiment with the Perceptron, we will use the Iris dataset, imported in the next cell.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame = True) # Load the iris dataset, including the pandas df form 
df_iris = load_iris(as_frame = True).frame # Extract the pandas df form
target_names = iris['target_names'] # Extract the target names

**Todo**: Insert a few new cells below to have a quick look at the dataset and find the following information: 
- The number of attributes in the dataset.
- The attributes' type(s).
- The total number of instances in the dataset.
- The number of instances per target category.
- The name of the target categories.

In [ ]:
df_iris.head()

In [ ]:
df_iris.info()

In [ ]:
target_names

In [ ]:
df_iris["target"].value_counts()

Once you found the required information, run the next cell to seperate the predictive attributes from the target. We want to keep the task simple by transforming it into a binary classification problem. The goal is to seperate the type *Iris setosa* from the 2 others, using the two predictive attributes `petal length` and `petal width`. 

In [ ]:
X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = (iris.target == 0)  # Iris setosa

Run the following cell to visualise the two attributes and the target classes.

In [ ]:
plt.plot(X[y == 0, 0], X[y == 0, 1], "bs", label="Not Iris setosa")
plt.plot(X[y == 1, 0], X[y == 1, 1], "yo", label="Iris setosa")
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.legend(loc="lower right")
plt.show()

Note that the two classes are linearly seperable, therefore we can use a single neuron Perceptron to learn a model to seperate them!

### I.3. Training and using a Perceptron for classification

You can import an implementation of the Perceptron from the Scikit-Learn library. Run the following cell to import the Perceptron class and declare a Perceptron *instance*.

**Note**:  Instance here is used in the object-oriented programming sense. It is an instance of the class Perceptron. This is not the same meaning then a *dataset instance*, which corresponds to a data point stored in a row of the dataset. 

In [ ]:
from sklearn.linear_model import Perceptron

perceptron_clf = Perceptron(random_state=42)

Notice the parameter `random_state` set to 42.  By setting the seed of the random number generator, the training instances will be processed in the same order every time the training is performed using this Perceptron's object. This is to allow repetability of the results.

Run the following cell to train the model. Feel free to have a look at the documentation the `fit()` method. It basically trains the model using Stochastic Gradient Descent (SGD). The model will be updated based on the rule defined in Equation (3), for each instance of the dataset. Note that we use the full data set to train the model here. Normally, we should split it in train and test set first. As this is a simple task with a small dataset, we decide not to do it, but we will in the following example using the next examples.

In [ ]:
perceptron_clf.fit(X, y)

Congratulations, you have trained your (maybe) first Perceptron!
You can now make new predictions using your trained model. Let's create 2 new data instances (i.e., 2 new flowers with attributes values not seen in the training) stored in `X_new`.

In [ ]:
X_new = [[2, 0.5], [3, 1]]

In [ ]:
plt.plot(X[y == 0, 0], X[y == 0, 1], "bs", label="Not Iris setosa")
plt.plot(X[y == 1, 0], X[y == 1, 1], "yo", label="Iris setosa")
plt.plot(X_new[0][0], X_new[0][1], "rx", label="New instance 1")
plt.plot(X_new[1][0], X_new[1][1], "r*", label="New instance 2")
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.legend(loc="lower right")
plt.show()

**Todo**: Discuss what you would expect the result of the prediction to be for each new instance. 

Run the following cell to use your trained Perceptron to classify the 2 new instances (i.e., predict their class based on the model), and verify your intuition.

In [ ]:
y_pred = perceptron_clf.predict(X_new) # predicts True and False for these 2 new instances

In [ ]:
y_pred

For the first new instance, the Perceptron classifier we trained predicts that the flower is an Iris setosa, and for the second one, it predicts that it is not. Does it confirm your intuition looking at the scatter plot?  

We can also visualise the decision boundaries of our trained Perceptron.  
Run the following cell to visualise the decision boundary. Do not hesitate to have a longer look at the code in this cell later to understand better how the plot is generated.

In [ ]:
from matplotlib.colors import ListedColormap

a = -perceptron_clf.coef_[0, 0] / perceptron_clf.coef_[0, 1]
b = -perceptron_clf.intercept_ / perceptron_clf.coef_[0, 1]
axes = [0, 7, 0, 2.6]
x0, x1 = np.meshgrid(
    np.linspace(axes[0], axes[1], 500).reshape(-1, 1),
    np.linspace(axes[2], axes[3], 200).reshape(-1, 1),
)
X_new = np.c_[x0.ravel(), x1.ravel()]
y_predict = perceptron_clf.predict(X_new)
zz = y_predict.reshape(x0.shape)
custom_cmap = ListedColormap(['#9898ff', '#fafab0'])

plt.figure(figsize=(7, 3))
plt.plot(X[y == 0, 0], X[y == 0, 1], "bs", label="Not Iris setosa")
plt.plot(X[y == 1, 0], X[y == 1, 1], "yo", label="Iris setosa")
plt.plot([axes[0], axes[1]], [a * axes[0] + b, a * axes[1] + b], "k-",
         linewidth=3)
plt.contourf(x0, x1, zz, cmap=custom_cmap)
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.legend(loc="lower right")
plt.axis(axes)
plt.show()

Any new instance in the yellow area (i.e., under the line) will be predicted as part of the *Iris setosa* class, while every instance in the purple area (i.e., above the the line) will be predicted as being part of the *Not Iris setosa* class. You can see that this decision boundary is linear, as expected with a single neuron Perceptron.

**Todo**: Modify the random seed set when declaring the variable `perceptron_clf` and run the cells up to the decision boundary visualisation again. Can you see the boundary moving? This is due to the fact that training  a Perceptron is a non-deterministic process. The trained model will be slightly different if the data instances are presented in a different order during the training. 

## II. From a single artificial neuron to a network of artificial neurons: Multi-Layer Perceptrons (MLP) and Artificial Neural Networks (ANN) 

To model more complex functions, it is possible to combine single Perceptrons together into layers. A Multi-Layer Perceptron (MLP) is composed of:
- One **input layer**: this layer consists of the input values. It actually does not contain any Perceptrons and does not perform any computations. It only passes the input values to the next layer. 
- One or more **hidden layer(s)**: an hidden layer consists of several units (i.e., Perceptrons) performing computations described by Equation (1), in parallel. Note that in a MLP, the activation function $g$ can be different from a step function.
- One **output layer**: this layer consists of one or several units which compute the output(s) of the network. It can be composed of one or more units depending on the task at hand (e.g., binary vs multi-class classification).

Layers closer to the input layer are called *lower layers*, and layers closer to the output layer are called *upper layers*. 

In a MLP, all the units in one layer (or inputs for the input layer) are connected to all the units in the next layer, as shown in image below. Each layer is *fully connected* with the next one. Also, at prediction time, the information is only passed forward in the network (from lower layers to upper layers). This is called a *feed forward* neural network.  

<figure>
    <center> <img src="img/MLP.jpg"  alt='mlp' width="300"  ><center/>
    <figcaption>A MLP with 2 inputs (+ 1 for the bias term), 1 hidden layer with 4 units, and 2 output units.</figcaption>
<figure/>


The MLP is a special type of *Artificial Neural Network* (ANN), with the specific characteristics described previously. We will study different other types of ANNs along this course.  
As discussed in Lecture 3, an ANN with more than 1 hidden layer is called a *Deep Neural Network*, which is the field of study of *Deep Learning*. 

### II.1. MLP for Regression using Scikit Learn

MLPs can be used for classification and regression tasks. 
Let's start by using an MLP to train a model on the *californian districts housing dataset* used in Tutorial 1. In this case, our MLP will have 1 output neuron to predict the median house price. 

To keep things simple for this tutorial, we will use a simplified version of the dataset, obtained throught the Scikit Learn's `fetch_california_housing` function. This version of the dataset has only numerical attributes (the attribute `ocean_proximity` was removed), and no missing values. Feel free to play around later with your previous notebook to run the regression MLP on the full dataset you pre-processed last week. 

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()

First, we need to split our dataset into training, validation (dev) and test sets. We use the Scikit Learn `train_test_split` function to first split the full dataset into training and test, and then the resulting training set into a final training set and a validation set.  
Remember that the training set is used to train the model parameters (i.e., the weights), the validation set is used to make intermediate choices (e.g., hyperparameter tuning), and the test set is used to validate the final version of the model. 

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42)

We use the Scikit-Learn's `MLPRegressor` class to declare a MLP for regression. This MLP uses ReLu activation function in the hidden layers, and a vairant of gradient descent called *Adam* to optimise the weights' values. We will discuss these in a few lectures, so you do not need to worry about them at the moment.


**Todo**: Before running the following cell, answer the following questions:
- How many hidden layers and how many units per layers are there in the declared MLP?
- How many input units are there in the input layer?
- How many output units are there in the output layer?
- What type of preprocessing is applied to the inputs before feeding them to the MPL?
- `MLPRegressor` uses the MSE loss function to optimise the weigths. Does it make sense for this task?

You can now run the following cell to preprocess the training set and use it to train a MLP. This will take a few seconds to run.

In [ ]:
mlp_reg = MLPRegressor(hidden_layer_sizes=[50, 50, 50], random_state=42)
pipeline = make_pipeline(StandardScaler(), mlp_reg)
pipeline.fit(X_train, y_train)

Run the following cell to visualise the loss curve, representing the loss value evaluated at the end of each training step (epoch). 

In [ ]:
pd.DataFrame(mlp_reg.loss_curve_).plot()
plt.xlabel("Epochs")
plt.ylabel("MSE")
plt.show()

Notice that it possible to include the MLP into a pipeline, following directly the pre-preprocessing steps. 

You can now evaluate your trained MLP on the validation set. 

In [ ]:
y_pred = pipeline.predict(X_valid)
rmse = mean_squared_error(y_valid, y_pred, squared=False)

In [ ]:
rmse

**Todo**: Discuss the validation RMSE value, knowing that no activation function is applied to the output layer.

**Important**: Note that RMSE is a common metric to use for regression, but you may want to use other metrics for regression in specific cases. For example, if you have a lot of outliers, the *Mean Absolute Error* (MAE) is better as it punishes less harshly large mistakes (which would be caused by outliers). The *Huber loss* would also be a good option in that case, as it punishes more smaller mistakes and less larger mistakes. 

### II.2. Storing your model/pipeline

Once you trained your model and you are happy of its performances, you might want to save it so you do not have to train it again when you want to reuse it.  
To save a model, and even the full pipeline with preprocessing, you can use the `joblib` library. The `dump()` function allows you to save your model/pipeline, and the `load()` allows you to load it back.  
You can also use the Python `pickle` module with similar functions. The syntax when using `pickle` is slightly different. You can refer to its documentation to learn more about it.

In [ ]:
import joblib
joblib.dump(pipeline, 'mlp_reg_pipeline.joblib')

In [ ]:
pipeline_loaded = joblib.load('mlp_reg_pipeline.joblib')

In [ ]:
pipeline_loaded

You can verify that the loaded model has the same performance as the one you stored.

In [ ]:
y_pred = pipeline_loaded.predict(X_valid)
rmse = mean_squared_error(y_valid, y_pred, squared=False)

In [ ]:
rmse

### II.3. MLP for classification using Keras/Tensorflow

Similarly to the `MLPRegressor` class for regression, Sckit-Learn also provides a `MLPClassifier` for classification. The syntax is very similar. As you have just experienced, building and training a standard MLP for regression or classification is very quick with Scikit-Learn. However, it only let you modify and play with a limited amount of features and hyperparameters. For example, the `MLPRegressor` class only supports MSE loss for the optimisation, so you cannot use MAE for example.  
Scikit-Learn classes are great for quick experimentation and learning, but when building bigger and more complex ANNs, people usually use Keras/TensorFlow or PyTorch.  

We will now explore how to use both to develop, train and evaluate a MLP for image classification.  

#### II.3.i. Fashion MNIST dataset

We will use the Fashion MNIST dataset, which is a collection of fashion items' images from  Zalando's article images. There are 70 000 grayscale images of 28x28 pixels each, associated with a label from 10 classes. This is one of the most used benchmark dataset to evaluate machine learning models. 

#### II.3.ii. Loading the dataset with Keras/TensorFlow

Let's first use Keras/TensorFlow to load the dataset and then design and train a MLP classifier on it.
There are a number of functions to load popular datasets in `tf.keras.datasets`. The Fashion MNIST dataset is already split for you between a training set (60,000 images) and a test set (10,000 images), but it can be useful to split the training set further to have a validation set. We'll use 55,000 images for training, and 5,000 for validation.

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]

The training set now contains 55,000 grayscale images, each 28x28 pixels.

In [ ]:
X_train.shape

Each pixel intensity is represented as a byte (0 to 255).

In [ ]:
X_train.dtype

Let's scale the pixel intensities down to the 0-1 range and convert them to floats, by dividing by 255 (this also converts them to float).

In [ ]:
X_train, X_valid, X_test = X_train / 255., X_valid / 255., X_test / 255.

Run the following cell to plot an image from the dataset using Matplotlib's `imshow()` function, with a 'binary' color map.

In [ ]:
plt.imshow(X_train[0], cmap="binary")
plt.axis('off')
plt.show()

The labels are the class IDs (represented as uint8), from 0 to 9.

In [ ]:
y_train

Let's store the corresponding class names so we can convert the class ID to the correpsonding name.

In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

**Todo**: What is the class of the first image we displayed previously?

Run the following cell to confirm your answer!

In [ ]:
class_names[y_train[0]]

Run the following cell to display a sample of the images from the dataset, associated with their label.

In [ ]:
n_rows = 4
n_cols = 10
plt.figure(figsize=(n_cols * 1.2, n_rows * 1.2))
for row in range(n_rows):
    for col in range(n_cols):
        index = n_cols * row + col
        plt.subplot(n_rows, n_cols, index + 1)
        plt.imshow(X_train[index], cmap="binary", interpolation="nearest")
        plt.axis('off')
        plt.title(class_names[y_train[index]])
plt.subplots_adjust(wspace=0.2, hspace=0.5)
plt.show()

Let's see if we can build a MLP to predict which image belong to which class. 

**Important**: When learning with images, we usually prefer to use other types of ANNs (e.g., Convolutional Neural Networks), which are more adapted to model information from an image structure. Howerver, the images in the Fashion MNIST dataset are small and easy to discriminate, so using a MLP (i.e., fully-connected feed-foward ANN) works already very well.
  

#### II.3.iii. Creating a model using the Keras/TensorFlow Sequential class

We can build a network using the `Sequential` class from Keras/TensorFlow. The `add` method allow you to add layers to the model. 

**Todo**: Have a look at the following cell and identify the following steps.
- Adding an input layer with input shape corresponding to the image shape in the Fashion MNIST dataset.
- Adding a preprocessing layer to reshape each input into a 1D array.
- Adding 2 fully-connected hidden layers with respectfully 100 and 300 hidden units, both using the ReLu activation function.
- Adding an fully-connected output layer with 10 units, using the softmax activation function.

In [ ]:
tf.random.set_seed(42) # To allow reproducibility
model = tf.keras.Sequential()
model.add(tf.keras.layers.InputLayer(input_shape=[28, 28]))
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(300, activation="relu"))
model.add(tf.keras.layers.Dense(100, activation="relu"))
model.add(tf.keras.layers.Dense(10, activation="softmax"))

Well done! You declared your (maybe) first MLP!

Here are a few extra notes to complete your observations:
- The `Sequential` class is appropriate to build models composed of a single stack of layers connected sequentially. This is not always the case as we will see in later lectures. This is also called the sequential API. 
- We specify the shape of the inputs we will pass to the input layer because Keras needs to know this to determine the shape of the connection weight matrix of the first hidden layer.
- In a MLP, each pixel of an image corresponds to an input's attribute, i.e., corresponds to one input $x_i$ in the input layer. Therefore, we need to reshape each image into a vector of length 28 $\times$ 28 = 784. We usually feed training inputs as batches (using batch gradient descent). For example, if we use a batch size of 32, the shape of the input matrix for a batch is [32, 28, 28]. When being processed in the `Flatten` layer, the input matrix is reshaed to [32, 784].
- A `Dense` layer is equivalent to a fully-connected layer with bias terms. We need to specify the number of units in the layer, so that Keras can determine the shape of the connection weight matrix of the next layer. We can also specify the type of activation function being used. In the two hidden layer of this model, we decide to use the ReLu function. This is a popular activation function for hidden layers in ANNs.
- The ouput layer is declared are a `Dense` layer but with only 10 units (1 for each target class/label), and a Softmax activation function. This is a popular activation function for output layers when you want to predict 1 class out of several possible ones, and the classes are mutually exclusive (it can only be one of them). 

Using the `add` method to add the layers one by one is not always convenient. You can also pass a list of layers when creating a `Sequential` model. You can also drop the `Input` layer and instead specify the input shape in the `Flatten` layer directly:

In [ ]:
tf.keras.backend.clear_session() # clear the session memory to avoid storing several different models
tf.random.set_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(300, activation="relu"),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

You can visualise a summary of your model using the `summary()` method.

In [ ]:
model.summary()

You can see the number of parameters (weights and bias terms) in the right-most column. Note that Dense layers usually have a lot of parameters! For example, the first hidden layer has 784 $\times$ 300 connection weights, plus 300 bias terms, which adds up to 235 500 parameters! In total, your model parameter count is 266 610!  
This means that the model has a lot of degrees of freedom, i.e., a lot of flexibilty to fit the training data. It runs the risk of overfitting, especially if we do not have a lot of tranining data. 

The parameters of a layer can be accessed with the `get_weights()` method.

In [ ]:
weights, biases = model.layers[1].get_weights() # returns weights and biases matrices for the first hidden layer as np.array 
weights

In [ ]:
biases

**Todo**: What are the shapes of the weights and biases matrices for the first hidden layer? Think about how many inputs there are and how many units. Try to answer by yourself first, and you can then display them using the `shape` field of the Numpy array.  
What about in the other layers?

In [ ]:
weights.shape

In [ ]:
biases.shape

Notice that the weight matrix is initialised randomly and the bias matrix is initialised to 0. We will discuss why this is important and different possible initialisation methods in a future lecture.

**Important note**: TensorFlow uses a specialised data structure called *tensors* to encode the inputs and outputs of a model, as well as the model's parameters. The name TensorFlow comes from the idea that these tensors *flow* from operations to operations. These tensors are very similar to NumPy's ndarray in the way that they usually are multidimentional arrays, but they can also hold simple scalars. One big advantage of tensors is that they can run on GPUs or other hardward accelerator. Therefore, they are used a lot in machine learning and AI implementations as they allow to train models effectively.  
You can read more about tensors used in TensorFlow [here](https://www.tensorflow.org/guide/tensor).
PyTorch has its own version of tensors. You can read more about it [here](https://pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html).

#### II.3.iv. Compling the model

After creating the model and before training it, we need to compile it, using the `compile()` method. When calling this method, you have to specify the type of loss function and optimiser you wish to use. Also, you can specify a range of metrics to be computed during training and evaluation. 

In [ ]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

Let's explain the choices for the `loss`, `optimizer` and `metrics` briefly.
1. We are using the `sparse_categorical_crossentropy` loss function. Categorical crossentropy is a common metric used for multi-class classification. It measures the difference between the true probability distribution of the classes and the predicted probability distribution. We use sparce categorical crossentropy here because we have sparce labels (i.e., for each data instance, there is only one target class index, from 0 to 9 in this case). If we had a target probability per class for each data instance (such as one-hot vectors, e.g., [0., 0., 0., 1., 0., 0., 0., 0., 0.] to represent class 3), then we would have to use the `categorical_crossentropy`. You can find a nice explanation about the different between the 2 in this [StackOverflow discussion](https://stackoverflow.com/questions/58565394/what-is-the-difference-between-sparse-categorical-crossentropy-and-categorical-c).  
There exists other loss functions adapted to different types of problems, such as the `binary_crossentropy` for binary classification problems. You would also need to change the output activation function to `sigmoid` (instead of `softmax`) for binary classification tasks.
2. The optimiser is set to Stochastic Gradient Descent (SGD).
3. We only compute the accuracy as an evaluation metric. It is a useful metric to measure for classifiers. However, computing additional metrics, such as the precision, recall and F1 score might also be useful, especially if you have unbalance in your data. We will have a look at an example at the end of this tutorial.

#### II.3.v. Training and evaluating the model

We are now ready to train the model. This is done by calling the `fit()` method, passing it the input training data features `X_train` and the target classes `y_train`, as well as the number of `epochs` (i.e., the number of training iterations). We also pass a tuple with the validation set to calculate the performance on this set on top of the training set (the validation set is not used for the training, only to test the model performance).  

The next cell will take up to a few minutes to run. You can monitor the loss and accuracy values computed on the training set and validation set for each epoch as the training goes.

In [ ]:
history = model.fit(X_train, y_train, epochs=30,
                    validation_data=(X_valid, y_valid))

The `fit()` method returns a `History` object containing 
- the training parameters (e.g., the number of epochs) in the field `params`,
- the list of epochs it went through in the field `epochs`, and
- a dictionary in the field `history` containing a record of training loss values and extra metrics values at successive epochs, as well as validation loss values and validation metrics values (if any validation set was given).

We can use this last field to plot the learning curve.  
Run the following cell to plot these curves for the model you just trained.

In [ ]:
pd.DataFrame(history.history).plot(
    figsize=(8, 5), xlim=[0, 29], ylim=[0, 1], grid=True, xlabel="Epoch",
    style=["r--", "r--.", "b-", "b-*"])
plt.legend(loc="lower left")
plt.show()

Note that `loss` and `accuracy` refer to the loss and accuracy computed on the training set, while `val_loss` and `val_accuracy` refer to the loss and accuracy computed on the validation set.

**Todo**: What can you say about your model and its training when looking at these curves? What can you say about bias and variance. Is your model overfitting?  

Once you have discussed your observations with your classmates, you can run the following cell to verify your insights.

In [ ]:
def read_insights(file):
    with open(file, "r") as file:
        print(file.read())
read_insights("do_not_read.txt")

If you are not satisfied with the performances of your model, you can decide to tune some hyperparameters (e.g., the learning rate, the number of neurons per layer, etc.) and evaluate several configurations using the performance on the validation set (sometimes called the development set). However, this is not the focus of this tutorial, but we will talk about it in Tutorial 3.  

Let's say that you are happy with the current model's validation accuracy. You can now evaluate the generalisation error on the test set before deciding to deploy your model in production, and/or publish some results.  
Run the following cell to evaluate your model on the test set.

In [ ]:
model.evaluate(X_test, y_test)

You can see that the test accuracy is very close to the validation accuracy. This is common to get a slightly lower test accuracy than validation accuray in practice, as you would have tuned the hyperparameters on the validation set and not on the test set.

#### II.3.vi. Making predictions using the trained model

You can now make predictions with your newly trained model, using the `predict()` method. We do not have new instance, so let's use the first 3 instances of the test set.

In [ ]:
X_new = X_test[:3]
y_proba = model.predict(X_new)
y_proba.round(2)

For each instance, the model predict a probability per class, from class 0 on the left to class 9 on the right. For example, for the first instance the model estimated that the probability of class 9 (ankle boot) is 98%, the probability of class 7 (sneakers) is 2%, the probability of class 5 (sandals) of 1%, and the probability of the other classes is negligible. It is very confident that this image is a ankle boot, with a slight chance that it might be sneakers.  
If you want to convert these probability to single class prediction (using the highest probability as the selected class), you can use the `argmax()` method. 

In [ ]:
y_pred = y_proba.argmax(axis=-1)
y_pred

Let's convert the class indexes into the class names.

In [ ]:
np.array(class_names)[y_pred]

We can verify now that the predicted class are correct, by displayed the expected classes for the 3 first instances of the test set.

In [ ]:
y_new = y_test[:3]
y_new

The predictions are correct!

You can also verify this by displaying the images with the predicted labels.

In [ ]:
plt.figure(figsize=(7.2, 2.4))
for index, image in enumerate(X_new):
    plt.subplot(1, 3, index + 1)
    plt.imshow(image, cmap="binary", interpolation="nearest")
    plt.axis('off')
    plt.title(class_names[y_pred[index]])
plt.subplots_adjust(wspace=0.2, hspace=0.5)
plt.show()

You now know how to build a MLP for classification with Keras/TensorFlow, and train it on the Fashion MNIST dataset.

Let's now have a look at how we can do the same thing with PyTorch. Note that we will go more quickly through the example with PyTorch and skip a few steps that we performed with Keras/TensorFlow (e.g., we will not use a validation set). This goal of the next part is really to give you a general overview of how to train a simple MLP in PyTorch.

#### II.3.vii. Saving and restoring a model
 Saving a trained model with Keras/TensorFlow is very simple. You can use the `save()` method as shown below.

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
path = 'my_keras_models'
isExist = os.path.exists(path)
if not isExist:
    os.makedirs(path)
model.save("my_keras_models\model_{}".format(timestamp), save_format="tf")

When saved with `save_format="tf"`, the model is stored using TensorFlow's *Saved Model* format: this is a directory (with the given name in the first argument of the `save()` method) containing several files and subdirectories. The *saved_model.pb* file contains the models architecture and logic. The *keras_metadata.pb* file contains extra information needed by Keras. The *variables* directory contains all parameters values (including weights and biases, and extra parameters such as the optimiser's parameters), possibily split across mutliple files if the model is very big.  

Alternatively, you can also use other storing formats such as HDF5 using `save_format="h5"`. 

If you wish to load a model, you can use the `load_model()` function, as shown below. 

In [ ]:
loaded_model = tf.keras.models.load_model("my_keras_models\model_{}".format(timestamp))

In [ ]:
loaded_model.summary()

In [ ]:
X_new = X_test[:3]
y_proba = loaded_model.predict(X_new)
y_proba.round(2)

You can also choose to save only the weights of the model using the `save_weights()` and `load_weights()` methods.

#### II.3.viii. Using callbacks

The `fit()` method accepts a `callbacks` argument you can use to specify a list of objects you want Keras to call before and after training, before and after each epoch, and even before and after processing each batch. For example, the `ModelCheckpoint` callback can be used to save checkpoints of your model at regular intervals during training (by default after each epoch).

In [ ]:
path = 'my_keras_models'
isExist = os.path.exists(path)
if not isExist:
    os.makedirs(path)
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint("my_keras_models\my_checkpoints",
                                                   save_weights_only=True)
history = model.fit(X_train, y_train, epochs=5, validation_data=(X_valid, y_valid), callbacks=[checkpoint_cb])

If you use a validation set to evaluate the performance of your model during training, you can set `save_best_only = True` when you create the `ModelCheckpoint` object, to save the model weights only when the performance on the validation set is the best so far. 

There are a number of other types of callbacks you can use. You can find the list on the [Keras Callbacks API page](https://keras.io/api/callbacks/). You can even create your own custom callbacks! You can find some examples [here](https://keras.io/guides/writing_your_own_callbacks/).

### II.4. MLP for classification using PyTorch

#### II.4.i. Loading the dataset with PyTorch

We already saw how to import the data from PyTorch in the last tutorial. We just added a pre-processing step here with image normalisation.

In [ ]:
from torch.utils.data import Dataset
from torchvision import datasets
from  torchvision import transforms 

transform = transforms.Compose(
    [transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))]) # Image normalisation

ds_train = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

ds_test = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

You can run the 2 following cells to see how many instances there are in the train and test sets.

In [ ]:
len(ds_train)

In [ ]:
len(ds_test)

Let's split the train set into train and validation sets.

In [ ]:
ds_train, ds_val = torch.utils.data.random_split(
        ds_train, [50000, 10000], generator=torch.Generator().manual_seed(1))

PyTorch provides the `DataLoader` class to facilitate data loading. Let's create `DataLoader` objects for our 3 sets.

In [ ]:
train_loader = torch.utils.data.DataLoader(ds_train, batch_size=1, shuffle=True)
val_loader = torch.utils.data.DataLoader(ds_val, batch_size=1, shuffle=False)
test_loader = torch.utils.data.DataLoader(ds_test, batch_size=1, shuffle=False)

Note that we use `batch_size=1` as we want to perform stochastic gradient descent.

The images are already scaled to a 0-1 range when loaded , so we do not have to scale them ourselves.  

Let's have a quick look at the first image.

In [ ]:
image, label = ds_train[0]
label

In [ ]:
image.shape

Note that the image's shape is [1, 28, 28] because it has 28$\times$28 pixels and 1 channel (it would be 3 for a RGB image).  
We need to reshape it to (28, 28) if we want to display it with `plt.imshow`.

In [ ]:
plt.imshow(image.reshape((28, 28)), cmap="binary")
plt.axis('off')
plt.show()

#### II.4.ii. Creating a model using the PyTorch

Note that PyTorch was developped with a deep object-oriented approach, closer to the principles of the Python language. It provide more flexibility than TensorFlow, which makes it great for research and prototyping. However, it might lead to less optimised models than TensorFlow. Therefore, there are multiple possible way to create a model with PyTorch. We will cover 2 of them in this tutorial.

The first approach is to define a class for your model, ihnerited from the `nn.Module` class.  
In the `__init__()` method, you define the layers of your network.  
In PyTorch, you can use the `Linear` layer to apply a linear transformation ($y=xA^T+b$) to the layer's incoming data. For each `Linear` layer, we need to specify the size of the input sample and the size of the output sample. Note that they are actually the size of the parameters' matrix for each layer.  
The activation function is not included in a `Linear` layer object, so it needs to be specified seperatly.  
We then create a `forward()` method which will be called to perform a forward pass in the network. The input tensor `x` will first be flattened, and then passed through the layers and activation functions in sequential order. 

In summary, the following sequence  of actions is executed when calling the `forward()` method:  
1. Flatten the input tensor,
2. Pass the result to layer `fc1` and then apply `relu`,
3. Pass the result to layer `fc2` and then apply `relu`,
4. Pass the result to layer `fc3` and then apply `softmax`,
5. The result is returned.

In [ ]:
from torch import nn, optim

class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 300)
        self.fc2 = nn.Linear(300, 100)
        self.fc3 = nn.Linear(100, 10)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax()
        
    def forward(self, x):
        # Make sure the input tensor is flattened
        x = x.view(x.shape[0], -1)
        
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.softmax(self.fc3(x), dim=1)
        
        return x

torch_model = Classifier()

The second approach is to use the `Sequential` API, imported from its `nn` module, which can also be used to create a model with specified layers. This is simpler than the previous approach, but provides less flexibility to design custom models. You can add layers in a similar fashion than with Kears/TensorFlow. However, note that you still need to add activation functions separately between your layers.  

In [ ]:
torch_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 300),
    nn.ReLU(),
    nn.Linear(300, 100),
    nn.ReLU(),
    nn.Linear(100, 10),
    nn.Softmax(dim=1)
)

In [ ]:
print(torch_model)

The `model` object we created with this second approach is pretty similar to the one we created before with the class definition approach. You can use one way or the other depending on your preference and if you need more or less design flexibility.  
An object created with the PyTorch `Sequential` API has a `forward()` method performing the forward pass in the same way we described previously.  
Note that you will actually not need to call the `forward()` method. It is done "automatically" when you execute the instruction `torch_model(input)`. 

We will measure the cross entropy loss between or predicted and target classes, and we will use the SGD algorithm for optimisation. Note that we need to pass the models parameters as input to the `SGD` object.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(torch_model.parameters())

#### II.4.iii. Training a model with PyTorch

PyTorch doesn’t have a dedicated function for model training and evaluation. Therefore, it is your responsibility to write the training loop.  

**Todo**: Have a look a the training loop for one epoch (i.e., one pass through the full dataset) defined in the `train_one_epoch()` function in the following cell (this is directly taken from the PyTorch website), and have a look a tthe code and comments to identify the different training steps.  
Once you are familiar with it, you can run it.

In [ ]:
def train_one_epoch(epoch_index):
    running_loss = 0.
    last_loss = 0.

    # Here, we use enumerate(training_loader) instead of
    # iter(training_loader) so that we can track the batch
    # index and do some intra-epoch reporting
    for i, data in enumerate(train_loader):
        # Every data instance is an input + label pair
        inputs, labels = data

        # Zero your gradients for every batch!
        optimizer.zero_grad()

        # Make predictions for this batch
        outputs = torch_model(inputs)

        # Compute the loss and its gradients
        loss = loss_fn(outputs, labels)
        loss.backward()

        # Adjust learning weights
        optimizer.step()

        # Gather data and report
        running_loss += loss.item()
        if i % 2000 == 1999:
            last_loss = running_loss / 2000 # loss per batch
            print('  batch {} loss: {}'.format(i + 1, last_loss))
            running_loss = 0.

    return last_loss

Note that the `train_one_epoch()` function also prints the current training loss after processing each batch of 2000 inputs.

The next cell is performing another loop to run several epochs. In addtion to running one epoch using the `train_one_epoch()` function, the code performs, once per epoch:
we’ll want to do once per epoch:
- Perform validation by checking the relative loss on a set of data that was not used for training, and print it,
- Save a copy of the model after each epoch if the validation loss is improved.

We will only run 5 epochs as they take significantly longer to run with this PyTorch implementation. 

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
# Initializing in a separate cell so we can easily add more epochs to the same run
epoch_number = 0

EPOCHS = 1

best_vloss = 1_000_000.

for epoch in range(EPOCHS):
    print('EPOCH {}:'.format(epoch_number + 1))

    # Make sure gradient tracking is on, and do a pass over the data
    torch_model.train(True)
    avg_loss = train_one_epoch(epoch_number)

    running_vloss = 0.0
    # Set the model to evaluation mode, disabling dropout (if used) and using population
    # statistics for batch normalization.
    torch_model.eval()

    # Disable gradient computation and reduce memory consumption.
    with torch.no_grad():
        for i, vdata in enumerate(val_loader):
            vinputs, vlabels = vdata
            voutputs = torch_model(vinputs)
            vloss = loss_fn(voutputs, vlabels)
            running_vloss += vloss

    avg_vloss = running_vloss / (i + 1)
    print('LOSS train {} valid {}'.format(avg_loss, avg_vloss))

    # Track best performance, and save the model's state
    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        path = 'my_pytorch_models'
        isExist = os.path.exists(path)
        if not isExist:
            os.makedirs(path)
        model_path = 'my_pytorch_models\model_{}_{}'.format(timestamp, epoch_number)
        torch.save(torch_model.state_dict(), model_path)

    epoch_number += 1

Note that if you want to calculate and display the accuracy during the training, as we did with Keras/Tensorflow, there is no pre-coded class or method to do it. You need to code it yourself. We will not cover it in this tutorial, but feel free to try to do it after the tutorial. This is a good exercise to better understand how this code works and how accuracy is computed! 

Remember that we saved a version of the model after each epoch.  
You can load a saved model with PyTorch, using the `torch.load()` function. E.g.:
```
saved_model = torch.load(PATH)
```

### II.5. MLP for Regression using a Sequential API

You know now how to create MLPs for classification with Keras/TensorFlow and PyTorch.  
We have seen before that MLPs can also be used on regression problems too. If you want to predict a single continuous value, you just need an output layer with a single unit, using a linear activation function (i.e., the value of the unit's output is the output of the weighted sum of its inputs). 

We covered earlier how to do this with Scikit-Learn's `MLPRegressor` class, but you can also use the Sequential APIs of Keras/TensorFlow or PyTorch to build a more customisable model. We will not cover this in this tutorial but you can find an example using Keras/TensorFlow in the [Chapter 10 notebook of the "*Hands on ML*" book](https://github.com/ageron/handson-ml3/blob/main/10_neural_nets_with_keras.ipynb).


## III Evaluation using a Confusion Matrix

To finish this tutorial, let's have a quick look at how we can use a confusion matrix to evaluate the performance of a model in Python.  
We will use the MNIST dataset, loaded from the OpenML repository, through Scikit-Learn.

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', as_frame=False)

You can run the following cell to learn more about this dataset.

In [ ]:
print(mnist.DESCR)

Feel free to do your own data exploration (e.g., print some images and labels) before moving forward. You can have a look at the [Chapter 3 notebook of the "Hands on ML" book](https://github.com/ageron/handson-ml3/blob/main/03_classification.ipynb) for some ideas.

Next, let's seperate the dataset into train and test sets.

In [ ]:
X, y = mnist.data, mnist.target
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

The MNIST dataset has 10 classes (0 to 9 digits). We want to simplify the problem by converting it to a binary classification problem. We can do this by selecting only one digit we want to detect, against all the others. Let's choose number 7.  

In [ ]:
y_train_7 = (y_train == '7')  # True for all 7s, False for all other digits
y_test_7 = (y_test == '7')

As we want to illustrate here the use of the confusion matrix, we will use a `DummyClassifier` which classify every input as the majority class (here, *non* 7). 

In [ ]:
from sklearn.dummy import DummyClassifier

dummy_clf = DummyClassifier()
dummy_clf.fit(X_train, y_train_7)
# any returns True is any element in the predictions on the train set is True, else it returns False
print(any(dummy_clf.predict(X_train))) 

**Todo**: Try to guess what will be the model accuracy, given what you know about the task and the MNIST dataset. Then, you can run the following cell.

In [ ]:
from sklearn.model_selection import cross_val_score
cross_val_score(dummy_clf, X_train, y_train_7, cv=3, scoring="accuracy") #3-fold cross-validation

Did you guess correctly? It actually make sense given that as only about 10% of the images in the dataset are 7s! Therefore, the dummy model is right in all the other cases as it always guesses that the image is *not* a 7.

This shows you in practice that a high accuracy value might not mean much, especially in a situation of class imbalance, which we have here as only 10% of the digits are 7s, and therefore 90% are *non* 7s. 

Let's compute the confusion matrix to break down the performance of the model, as well as the precision, recall and f1-score.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_train_pred = cross_val_predict(dummy_clf, X_train, y_train_7, cv=3) #3-fold cross-validation predictions
cm = confusion_matrix(y_train_7, y_train_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=dummy_clf.classes_)
disp.plot()
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
precision_score(y_train_7, y_train_pred) # or cm[1, 1] / (cm[0, 1] + cm[1, 1])

In [ ]:
recall_score(y_train_7, y_train_pred) # or cm[1, 1] / (cm[1, 0] + cm[1, 1])

In [ ]:
f1_score(y_train_7, y_train_pred)  # cm[1, 1] / (cm[1, 1] + (cm[1, 0] + cm[0, 1]) / 2)

You see that we have 90% accuracy, but 0% recall, precision and f1-score! This is of course an extreme case, but it illustrate what we discussed in class about accuracy, especially in a scenario with class imbalance.  
Note that you can also calculate them using `classification report` library from `sklearn.metrics`.

**Todo**: Train a single Perceptron on this same task, and generate the Confusion matrix and associated metrics. Discuss the results.

In [ ]:
perceptron_clf2 = Perceptron()

In [ ]:
perceptron_clf2.fit(X_train, y_train_7)

In [ ]:
cross_val_score(perceptron_clf2 , X_train, y_train_7, cv=3, scoring="accuracy") #3-fold cross-validation

In [ ]:
y_train_pred2 = cross_val_predict(perceptron_clf2, X_train, y_train_7, cv=3) #3-fold cross-validation predictions
cm2 = confusion_matrix(y_train_7, y_train_pred2)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm2,
                              display_labels=perceptron_clf2.classes_)
disp2.plot()
plt.show()

In [ ]:
precision_score(y_train_7, y_train_pred2)

In [ ]:
recall_score(y_train_7, y_train_pred2)

In [ ]:
f1_score(y_train_7, y_train_pred2)

**To continue**: Build a small MLP on the MNIST dataset with all 10 classes and evaluate the trained model for each seperate class. 